In [5]:
# -*- coding: utf-8 -*-
import os, re, glob
import numpy as np, pandas as pd
from snownlp import SnowNLP
from tqdm import tqdm

base_path  = r"D:\研\大论文\数据\评论数据（原数据＋预处理后数据）\预处理后数据"
stage_path = r"D:\研\大论文\数据\热度数据\分年份数据\Fisher分段结果汇总.xlsx"
label_path = r"D:\研\大论文\数据\热度数据\分年份数据\4分类.xlsx"

# 输出路径设置
out_dir    = r"D:\研\大论文\数据\热度数据\指标_情感分析"
out_event_dir = os.path.join(out_dir, "事件单独结果")
out_long_csv  = os.path.join(out_dir, "情感分析_评论总表.csv")
out_sum_xlsx  = os.path.join(out_dir, "情感分析_总表.xlsx")
out_err_xlsx  = os.path.join(out_dir, "情感分析_失败记录.xlsx")
os.makedirs(out_event_dir, exist_ok=True)

years  = ['2021','2022','2023','2024','2025','2026']
stages = ['爆发期','成熟期','衰退期']

def clean_name(x):
    x = str(x).strip()
    x = re.sub(r'^微博源数据[-—]+', '', x).strip()
    x = re.sub(r'_小时数据$', '', x).strip()
    # 移除 Windows 文件名非法字符
    return re.sub(r'[\\/*?:"<>|]', '_', x)

def sentiment(t):
    if pd.isna(t) or str(t).strip() == '': return np.nan
    try: return SnowNLP(str(t)).sentiments
    except: return np.nan

def classify(v):
    if pd.isna(v): return np.nan
    return '消极' if v < 0.33 else ('中性' if v < 0.66 else '积极')

def find_col(cols, cands):
    for c in cands:
        for col in cols:
            if c == str(col).strip(): return col
    for c in cands:
        for col in cols:
            if c in str(col): return col
    return None

def read_any(f):
    for enc in ['utf-8-sig','gb18030','gbk','utf-8']:
        try:
            return pd.read_csv(f, encoding=enc) if f.endswith('.csv') else pd.read_excel(f)
        except UnicodeDecodeError:
            continue
    return None

# 计算单个事件（或阶段）的情感指标
def calc_metrics(g):
    s = g['情感得分'].dropna()
    c = g['情感分类'].dropna()
    n = len(s)
    p = c.value_counts(normalize=True)
    pn, pu, pp = p.get('消极',0.0), p.get('中性',0.0), p.get('积极',0.0)
    ps = np.array([pn,pu,pp]); ps = ps[ps>0]
    ent = float(-(ps*np.log(ps)).sum()/np.log(3)) if len(ps)>1 else 0.0
    return pd.Series({'评论数':n,'情感均值':s.mean(),'情感标准差':s.std(),
                      '消极占比':pn,'中性占比':pu,'积极占比':pp,
                      '情感熵':ent,'极化指数':2*min(pn,pp)})

# ---------- 读表 ----------
lab = pd.read_excel(label_path)
lab['年份'] = lab['年份'].astype(str).str.strip()
lab['_key'] = lab['事件'].apply(clean_name)
label_map = {(r['年份'], r['_key']): r['类别4'] for _, r in lab.iterrows()}

stage = pd.read_excel(stage_path)
stage['年份'] = stage['年份'].astype(str).str.strip()
stage['事件'] = stage['事件名称'].apply(clean_name)
stage['hour'] = stage['小时序号'].astype(int)
stage_map = stage[['年份','事件','hour','阶段类型']]

# ---------- 遍历处理 ----------
all_long, all_sum, errors = [], [], []

for y in years:
    folder = os.path.join(base_path, y)
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, "*.*"))
    
    for f in tqdm(files, desc=f"{y}年", unit="file", ncols=100):
        if not f.lower().endswith(('.csv','.xlsx','.xls')): continue
        name = os.path.splitext(os.path.basename(f))[0]
        key = clean_name(name)
        
        if (y, key) not in label_map: continue
        
        try:
            df = read_any(f)
            if df is None: raise ValueError("文件读取失败")
            
            cols = df.columns.tolist()
            tc = find_col(cols, ['时间','发布时间','日期','小时序号'])
            xc = find_col(cols, ['微博内容_清洗后','微博内容','评论内容','评论','内容'])
            if tc is None or xc is None:
                raise ValueError(f"缺时间/正文列: {cols}")
                
            df['_dt'] = pd.to_datetime(df[tc], errors='coerce')
            df = df.dropna(subset=['_dt']).reset_index(drop=True)
            if len(df) == 0: raise ValueError("时间全部解析失败")
            
            df['hour'] = ((df['_dt'] - df['_dt'].min()).dt.total_seconds()/3600).astype(int)
            df['情感得分'] = df[xc].apply(sentiment)
            df['情感分类'] = df['情感得分'].apply(classify)

            # 1. 生成该事件的评论明细 (Sheet1)
            tmp = pd.DataFrame({
                '年份': y, '事件名称': key, '类别4': label_map[(y,key)],
                '评论序号': np.arange(len(df)), '评论时间': df['_dt'],
                '小时序号': df['hour'], '情感得分': df['情感得分'],
                '情感分类': df['情感分类'], '正文': df[xc].astype(str).str.slice(0, 200)
            })
            tmp = tmp.merge(stage_map, left_on=['年份','事件名称','小时序号'],
                            right_on=['年份','事件','hour'], how='left')
            tmp.drop(columns=['事件','hour'], inplace=True, errors='ignore')
            
            # 2. 计算该事件的汇总指标 (Sheet2)
            overall = calc_metrics(tmp)
            res_row = {'年份': y, '事件名称': key, '类别4': label_map[(y,key)]}
            for k, v in overall.items(): res_row[f'整体_{k}'] = v
            
            for s in stages:
                sub = tmp[tmp['阶段类型'] == s]
                stg_m = calc_metrics(sub) if len(sub) > 0 else calc_metrics(pd.DataFrame(columns=['情感得分','情感分类']))
                for k, v in stg_m.items(): res_row[f'{s}_{k}'] = v
                
            ev_df = pd.DataFrame([res_row])

            # 3. 导出该事件单独的 Excel
            excel_name = re.sub(r'[\\/*?:"<>|]', '_', f"{y}_{key}.xlsx")
            save_path = os.path.join(out_event_dir, excel_name)
            with pd.ExcelWriter(save_path, engine='openpyxl') as writer:
                ev_df.to_excel(writer, sheet_name='事件汇总', index=False)
                tmp.to_excel(writer, sheet_name='评论明细', index=False)

            # 4. 收集到总表
            all_long.append(tmp)
            all_sum.append(ev_df)

        except Exception as e:
            errors.append({'年份': y, '事件名称': key, '错误信息': str(e)})

# ---------- 导出总表与错误记录 ----------
print("\n✅ 处理完成，正在导出汇总表...")
if all_long:
    pd.concat(all_long, ignore_index=True).to_csv(out_long_csv, index=False, encoding='utf-8-sig')
if all_sum:
    pd.concat(all_sum, ignore_index=True).to_excel(out_sum_xlsx, index=False)
if errors:
    pd.DataFrame(errors).to_excel(out_err_xlsx, index=False)
    print(f"⚠️ 有 {len(errors)} 个事件失败，请查看: {out_err_xlsx}")

print(f"✅ 事件单独结果存至: {out_event_dir}")
print(f"✅ 评论级长表存至: {out_long_csv}")
print(f"✅ 事件级总表存至: {out_sum_xlsx}")

2026年: 100%|███████████████████████████████████████████████████████| 5/5 [02:55<00:00, 35.03s/file]



✅ 处理完成，正在导出汇总表...
✅ 事件单独结果存至: D:\研\大论文\数据\热度数据\指标_情感分析\事件单独结果
✅ 评论级长表存至: D:\研\大论文\数据\热度数据\指标_情感分析\情感分析_评论总表.csv
✅ 事件级总表存至: D:\研\大论文\数据\热度数据\指标_情感分析\情感分析_总表.xlsx


In [6]:
# -*- coding: utf-8 -*-
import os
import pandas as pd

# 1. 路径设置（读取上一步生成的总表）
input_path = r"D:\研\大论文\数据\热度数据\指标_情感分析\情感分析_总表.xlsx"
out_path1  = r"D:\研\大论文\数据\热度数据\指标1_情感.xlsx"
out_path2  = r"D:\研\大论文\数据\热度数据\指标2_情感冲突.xlsx"

# 2. 读取数据
if not os.path.exists(input_path):
    raise FileNotFoundError(f"找不到总表，请确认路径: {input_path}")
    
df = pd.read_excel(input_path)
print(f"✅ 成功读取总表: {len(df)} 行 × {len(df.columns)} 列")

# 3. 基础标识列
id_cols = ['年份', '事件名称', '类别4']

# 4. 挑选列名
# 指标1：均值、消极占比、中性占比、积极占比
cols_sent = [c for c in df.columns if '情感均值' in c or '占比' in c]

# 指标2：标准差、情感熵、极化指数
cols_conf = [c for c in df.columns if '标准差' in c or '熵' in c or '极化' in c]

# 5. 生成两个子表
df_sent = df[id_cols + cols_sent].copy()
df_conf = df[id_cols + cols_conf].copy()

# 6. 导出
os.makedirs(os.path.dirname(out_path1), exist_ok=True)
df_sent.to_excel(out_path1, index=False)
df_conf.to_excel(out_path2, index=False)

print(f"✅ 指标1_情感 已导出: {len(df_sent.columns)} 列 → {out_path1}")
print(f"✅ 指标2_情感冲突 已导出: {len(df_conf.columns)} 列 → {out_path2}")

✅ 成功读取总表: 213 行 × 35 列
✅ 指标1_情感 已导出: 19 列 → D:\研\大论文\数据\热度数据\指标1_情感.xlsx
✅ 指标2_情感冲突 已导出: 15 列 → D:\研\大论文\数据\热度数据\指标2_情感冲突.xlsx
